In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ==========================================================
# Read Silver Tables
# ==========================================================

inpatient_df = spark.read.table(
    "healthcare_claims_catalog.silver.inpatient"
)

outpatient_df = spark.read.table(
    "healthcare_claims_catalog.silver.outpatient"
)

carrier_df = spark.read.table(
    "healthcare_claims_catalog.silver.carrier"
)

# ==========================================================
# Diagnosis Columns
# ==========================================================

inpatient_diag_cols = [
    "ADMTNG_ICD9_DGNS_CD",
    "ICD9_DGNS_CD_1",
    "ICD9_DGNS_CD_2",
    "ICD9_DGNS_CD_3",
    "ICD9_DGNS_CD_4",
    "ICD9_DGNS_CD_5",
    "ICD9_DGNS_CD_6",
    "ICD9_DGNS_CD_7",
    "ICD9_DGNS_CD_8",
    "ICD9_DGNS_CD_9",
    "ICD9_DGNS_CD_10"
]

outpatient_diag_cols = [
    "ADMTNG_ICD9_DGNS_CD",
    "ICD9_DGNS_CD_1",
    "ICD9_DGNS_CD_2",
    "ICD9_DGNS_CD_3",
    "ICD9_DGNS_CD_4",
    "ICD9_DGNS_CD_5",
    "ICD9_DGNS_CD_6",
    "ICD9_DGNS_CD_7",
    "ICD9_DGNS_CD_8",
    "ICD9_DGNS_CD_9",
    "ICD9_DGNS_CD_10"
]

carrier_diag_cols = [
    "ICD9_DGNS_CD_1",
    "ICD9_DGNS_CD_2",
    "ICD9_DGNS_CD_3",
    "ICD9_DGNS_CD_4",
    "ICD9_DGNS_CD_5",
    "ICD9_DGNS_CD_6",
    "ICD9_DGNS_CD_7",
    "ICD9_DGNS_CD_8",
    "LINE_ICD9_DGNS_CD_1",
    "LINE_ICD9_DGNS_CD_2",
    "LINE_ICD9_DGNS_CD_3",
    "LINE_ICD9_DGNS_CD_4",
    "LINE_ICD9_DGNS_CD_5",
    "LINE_ICD9_DGNS_CD_6",
    "LINE_ICD9_DGNS_CD_7",
    "LINE_ICD9_DGNS_CD_8",
    "LINE_ICD9_DGNS_CD_9",
    "LINE_ICD9_DGNS_CD_10",
    "LINE_ICD9_DGNS_CD_11",
    "LINE_ICD9_DGNS_CD_12",
    "LINE_ICD9_DGNS_CD_13"
]

# ==========================================================
# Extract Diagnosis Codes
# ==========================================================

inpatient_diag = inpatient_df.select(
    explode(array(*[col(c) for c in inpatient_diag_cols])).alias("DIAGNOSIS_CODE")
)

outpatient_diag = outpatient_df.select(
    explode(array(*[col(c) for c in outpatient_diag_cols])).alias("DIAGNOSIS_CODE")
)

carrier_diag = carrier_df.select(
    explode(array(*[col(c) for c in carrier_diag_cols])).alias("DIAGNOSIS_CODE")
)

# ==========================================================
# Combine All Diagnosis Codes
# ==========================================================

diagnosis_df = (
    inpatient_diag
    .unionByName(outpatient_diag)
    .unionByName(carrier_diag)
)

# ==========================================================
# Clean Diagnosis Codes
# ==========================================================

diagnosis_df = (
    diagnosis_df
    .withColumn("DIAGNOSIS_CODE", trim(col("DIAGNOSIS_CODE")))
    .filter(col("DIAGNOSIS_CODE").isNotNull())
    .filter(col("DIAGNOSIS_CODE") != "")
    .dropDuplicates(["DIAGNOSIS_CODE"])
)

# ==========================================================
# Create Surrogate Key
# ==========================================================

window_spec = Window.orderBy("DIAGNOSIS_CODE")

diagnosis_df = diagnosis_df.withColumn(
    "DIAGNOSIS_KEY",
    row_number().over(window_spec)
)

# ==========================================================
# Audit Column
# ==========================================================

diagnosis_df = diagnosis_df.withColumn(
    "GOLD_CREATED_TIMESTAMP",
    current_timestamp()
)

# ==========================================================
# Final Column Order
# ==========================================================

dim_diagnosis = diagnosis_df.select(
    "DIAGNOSIS_KEY",
    "DIAGNOSIS_CODE",
    "GOLD_CREATED_TIMESTAMP"
)

# ==========================================================
# Write Gold Table
# ==========================================================

dim_diagnosis.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "healthcare_claims_catalog.gold.dim_diagnosis"
    )

# ==========================================================
# Summary
# ==========================================================

print("=" * 60)
print("Gold Dimension - Diagnosis Created Successfully")
print("=" * 60)

print("Total Diagnosis Codes :", dim_diagnosis.count())

dim_diagnosis.printSchema()

dim_diagnosis.show(20, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold Dimension - Diagnosis Created Successfully
Total Diagnosis Codes : 13420
root
 |-- DIAGNOSIS_KEY: integer (nullable = false)
 |-- DIAGNOSIS_CODE: string (nullable = true)
 |-- GOLD_CREATED_TIMESTAMP: timestamp (nullable = false)

+-------------+--------------+--------------------------+
|DIAGNOSIS_KEY|DIAGNOSIS_CODE|GOLD_CREATED_TIMESTAMP    |
+-------------+--------------+--------------------------+
|1            |0010          |2026-08-06 17:19:25.229194|
|2            |0011          |2026-08-06 17:19:25.229194|
|3            |0019          |2026-08-06 17:19:25.229194|
|4            |0020          |2026-08-06 17:19:25.229194|
|5            |0021          |2026-08-06 17:19:25.229194|
|6            |0022          |2026-08-06 17:19:25.229194|
|7            |0023          |2026-08-06 17:19:25.229194|
|8            |0029          |2026-08-06 17:19:25.229194|
|9            |0030          |2026-08-06 17:19:25.229194|
|10           |0031          |2026-08-06 17:19:25.229194|
|11        